# Practical PyTorch · II — Part 5 — Companion Notebook

This goes with **"Fine-Tuning the Lazy Way: Let the Trainer Run Your Loop"**. Run the cells top to bottom (Shift+Enter).

**Turn the GPU on first:** **Runtime → Change runtime type → Hardware accelerator → GPU**. Fine-tuning on a CPU is painfully slow — with a GPU this whole notebook runs in a few minutes.

## 0. Setup

Colab ships with PyTorch, but we need the Hugging Face libraries. This installs them quietly.

In [ ]:
!pip install -q transformers datasets

import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU — turn it on: Runtime → Change runtime type → GPU")

## 1. Load a dataset (and keep it small)

We'll use **IMDB**: 50,000 movie reviews, each labeled positive (1) or negative (0). The whole thing would take too long on a free GPU, so we take a small slice of each split.

`shuffle` matters — IMDB ships sorted by label, so without it you'd grab all-negative reviews and train a useless model.

In [ ]:
from datasets import load_dataset

ds = load_dataset("imdb")
print(ds)

train_ds = ds["train"].shuffle(seed=0).select(range(2000))
eval_ds  = ds["test"].shuffle(seed=0).select(range(500))

print("\ntrain:", len(train_ds), "| eval:", len(eval_ds))
print("example label:", train_ds[0]["label"])
print("example text :", train_ds[0]["text"][:200], "...")

## 2. Tokenize with `.map`

A model reads numbers, not text. The tokenizer turns each review into token IDs. We run it over the whole dataset with `.map(..., batched=True)` — the `batched=True` makes it process chunks at once, which is much faster.

`truncation=True` clips reviews that exceed the model's maximum length.

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tok_fn(batch):
    return tok(batch["text"], truncation=True)

train_ds = train_ds.map(tok_fn, batched=True)
eval_ds  = eval_ds.map(tok_fn, batched=True)

print(train_ds.column_names)

## 3. The model, with a fresh classification head

`AutoModelForSequenceClassification` loads DistilBERT and bolts on a small, untrained classification head. `num_labels=2` because we have two classes (positive / negative).

You'll see a warning that some weights are "newly initialized" — that's the new head, and it's exactly what we're about to train. Not an error.

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=2,
)

## 4. TrainingArguments — the dials

This object holds every knob: how long to train, batch sizes, where to save, when to evaluate.

**Version note:** the argument is `eval_strategy` in recent `transformers`, but was `evaluation_strategy` in older versions. If you get a `TypeError` about an unexpected keyword, swap one for the other.

In [ ]:
from transformers import TrainingArguments

args = TrainingArguments(
    output_dir="out",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",      # older transformers: evaluation_strategy="epoch"
    logging_steps=50,
    report_to="none",
)

## 5. (Optional) a metric, so we can read accuracy

The Trainer reports loss by default. A tiny `compute_metrics` function also gives us **accuracy** after each epoch, which is far easier to interpret. (Part 6 goes deep on evaluation — this is just a taste.)

In [ ]:
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = (preds == labels).mean()
    return {"accuracy": acc}

## 6. The Trainer, then `train()`

Hand the Trainer the model, the arguments, both datasets, and the tokenizer — then call `train()`. This is the Part 2 loop (forward → loss → backward → step) plus batching, evaluation, logging, and checkpoints, all done for you.

A few-minute run on a GPU.

**Version note:** `tokenizer=tok` still works but is deprecated in newer `transformers`, which prefers `processing_class=tok`. Use whichever your version accepts.

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tok,                 # newer transformers: processing_class=tok
    compute_metrics=compute_metrics,
)

trainer.train()

## 7. Try it out

`model` is no longer generic DistilBERT — it's *your* sentiment classifier. Let's run a couple of reviews through it. (We tokenize the input the same way we tokenized training data, move it to the GPU, and read off the predicted class.)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

reviews = [
    "An absolute masterpiece. I'd watch it again tomorrow.",
    "Two hours of my life I will never get back. Painful.",
]

inputs = tok(reviews, padding=True, truncation=True, return_tensors="pt").to(device)
with torch.no_grad():
    logits = model(**inputs).logits

labels = {0: "NEGATIVE", 1: "POSITIVE"}
preds = logits.argmax(dim=-1).tolist()
for r, p in zip(reviews, preds):
    print(f"{labels[p]:>8}  |  {r}")

That's a full fine-tune: load a dataset, tokenize it with `.map`, pick a model with the right `num_labels`, set a few `TrainingArguments`, and let the `Trainer` run the loop you wrote by hand in Part 2.

**Next:** Part 6 — *Evaluating Your Model*, where we stop trusting the loss and measure whether the classifier actually works.